In [ ]:
import random
import numpy as np
import time
import numba
import math
import concurrent.futures
from numpy.random import Generator, PCG64DXSM, SeedSequence
from structure_MODIFIED_FNs_singlesideBASEreinLEARNING2_2026 import play_sequence

np.set_printoptions(suppress=True)

In [2]:
# use this cell for setting initial perameters

sides = 2
pairs = np.asarray([[0, 1], [1, 0], [1, 2], [2, 1], [2, 3], [3, 2], [3, 4], [4, 3]])
testpairs = np.asarray([[1, 3], [3, 1]])
nonadjpairs = np.asarray([[0, 2], [2, 0], [0, 3], [3, 0], [0, 4], [4, 0], [1, 4], [4, 1], [2, 4], [4, 2]])
allpairs = np.concatenate((pairs, nonadjpairs, testpairs))

pairs_reward = np.asarray([[1, 0] if a > b else [0, 1] for a, b in pairs], dtype=np.int64)
testpairs_reward = np.asarray([[1, 0] if a > b else [0, 1] for a, b in testpairs], dtype=np.int64)
nonadjpairs_reward = np.asarray([[1, 0] if a > b else [0, 1] for a, b in nonadjpairs], dtype=np.int64)
allpairs_reward = np.concatenate((pairs_reward, nonadjpairs_reward, testpairs_reward))

plen = len(pairs)
alen = len(allpairs)

terms = 5
maxvalue = 10
startstop = 2
noise = 0.
annealing = 0.

timesteps = 10**8
runs = 1000

rein1 = 4
rein2 = 4
punish1 = -11
punish2 = -11
inertia = 1

nsteps = 100
blocklength = nsteps*1


In [3]:
# use this cell for running the code

start = time.perf_counter()
thrds = 3

sq1 = SeedSequence()
randomentropy = sq1.entropy
sg = SeedSequence(randomentropy)
rgs = [Generator(PCG64DXSM(s)) for s in sg.spawn(runs)]

final_sigweights = [None] * runs
final_cumsuc = [None] * runs
final_cumsucadd = [None] * runs
final_testcumsucadd = [None] * runs
final_recweights = [None] * runs
final_pair_stats = [None] * runs

with concurrent.futures.ProcessPoolExecutor(max_workers=thrds) as executor:
    future_to_run = {executor.submit(play_sequence, r, rgs[r], rein1, punish1, rein2, punish2, timesteps, nsteps, sides, pairs, testpairs, nonadjpairs, allpairs, pairs_reward, testpairs_reward, nonadjpairs_reward, allpairs_reward, plen, alen, terms, maxvalue, startstop, noise, annealing, runs, inertia, blocklength): r for r in range(runs)}
    for future in concurrent.futures.as_completed(future_to_run):
        r = future_to_run[future]
        try:
            res = future.result()
            final_sigweights[r] = res[0]
            final_cumsuc[r] = res[1]
            final_cumsucadd[r] = res[2]
            final_testcumsucadd[r] = res[3]
            final_recweights[r] = res[4]
            final_pair_stats[r] = res[5]
            if r % 10 == 0:
                print(f'finished run #{r}')
        except Exception as exc:
            print(f'generated an exception in run {r}: {exc}')

final_sigweights = np.asarray(final_sigweights)
final_cumsuc = np.asarray(final_cumsuc)
final_cumsucadd = np.asarray(final_cumsucadd)
final_testcumsucadd = np.asarray(final_testcumsucadd)
final_recweights = np.asarray(final_recweights)
final_pair_stats = np.asarray(final_pair_stats)

print("average cumsuc = ")
print(np.average(final_cumsuc)/runs)
print(" ")
print("average final cumsucadd = ")
print(np.average(final_cumsucadd)/(100))
print(" ")
print("average test cumsum = ")
print(np.average(final_testcumsucadd)/(100))
print(" ")

finish = time.perf_counter()
print(f'Finished in {round(finish-start,0)/60} minutes')

In [4]:
np.save('structure_noiseAnn_dsktp_2s_BASElearning-MV10_MODIFIED_sigweights', final_sigweights)
np.save('structure_noiseAnn_dsktp_2s_BASElearning-MV10_MODIFIED_recweights', final_recweights)
np.save('structure_noiseAnn_dsktp_2s_BASElearning-MV10_MODIFIED_testcumsucadd', final_testcumsucadd)
np.save('structure_noiseAnn_dsktp_2s_BASElearning-MV10_MODIFIED_pair_stats', final_pair_stats)


In [5]:
distances = np.abs(testpairs[:, 0] - testpairs[:, 1])
unique_dist = np.unique(distances)
print("Success rate by distance:")
for d in unique_dist:
    mask = (distances == d)
    pair_indices = np.where(mask)[0]
    total_attempts = 0
    total_successes = 0
    for idx in pair_indices:
        total_attempts += np.sum(final_pair_stats[:, idx, 0])
        total_successes += np.sum(final_pair_stats[:, idx, 1])
    print(f"Distance {d}: {total_successes/total_attempts:.4f}")

In [6]:
@numba.njit
def duplicates(dbins):
    dup = 0
    for iii in range(len(dbins)):
        for jjj in range(len(dbins)):
            if iii != jjj:
                if (dbins[iii][0] == dbins[jjj][0]) & (dbins[iii][1] == dbins[jjj][1]):
                    dup = 1
    return dup

@numba.njit
def runs_dup_bins(runs, fsigweights, t):
    dup_count = 0
    for i0 in range(runs):
        sw = fsigweights[i0].copy()
        swl = sw.argmax(axis=3)[0, 0]
        swr = sw.argmax(axis=3)[0, 1]
        bns = np.zeros(((2*(t-1)), 2), dtype = numba.int64)
        for idx000 in range(t-1):
            bns[idx000][0] = swl[idx000] 
            bns[idx000][1] = swr[idx000+1]
        for idx001 in range(t-1):
            bns[t-1+idx001][0] = swl[idx001+1] 
            bns[t-1+idx001][1] = swr[idx001]
        dup_count += duplicates(bns)
    return dup_count

In [7]:
runs_dup_bins(runs, final_sigweights, terms)